# Chương 3: Thực hành - Biểu diễn tri thức và Lập luận (Knowledge Representation & Reasoning)

**Mục tiêu bài thực hành:**
1. Cài đặt cỗ máy **Suy diễn tiến (Forward Chaining)** và **Suy diễn lùi (Backward Chaining)** trên cơ sở tri thức mệnh đề Horn từ đầu (**from scratch**).
2. Cài đặt thuật toán **Hợp giải mệnh đề (Propositional Resolution Theorem Prover)** để chứng minh mệnh đề bằng phương pháp phản chứng.
3. Xây dựng **Đồ thị tri thức (Knowledge Graph)** dựa trên các bộ ba $(h, r, t)$ và thực hiện truy vấn suy luận nhiều bước (**Multi-hop Reasoning**).
4. Trực quan hóa cấu trúc đồ thị tri thức ngữ nghĩa.

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict, deque

## 1. Cơ sở tri thức mệnh đề Horn và Động cơ Suy diễn tiến (Forward Chaining)

Một mệnh đề Horn có dạng: $\text{Premise}_1 \land \text{Premise}_2 \land \dots \implies \text{Conclusion}$.  
Ta xây dựng một hệ thống chẩn đoán lỗi mạng máy tính (IT Network Troubleshooting).

In [ ]:
class HornClause:
    def __init__(self, premises, conclusion):
        self.premises = set(premises)  # Tập các điều kiện tiên quyết
        self.conclusion = conclusion    # Hệ quả suy ra
        
    def __repr__(self):
        if not self.premises:
            return f"Fact({self.conclusion})"
        return f"({' AND '.join(sorted(self.premises))}) => {self.conclusion}"

class ForwardChainingEngine:
    def __init__(self):
        self.clauses = []
        self.facts = set()
        
    def add_fact(self, fact):
        self.facts.add(fact)
        
    def add_rule(self, premises, conclusion):
        self.clauses.append(HornClause(premises, conclusion))
        
    def infer(self, query=None):
        """
        Thực thi suy diễn tiến: Lặp lại việc kích hoạt các luật có tiền đề đã thỏa mãn.
        """
        known_facts = set(self.facts)
        count = {i: len(clause.premises) for i, clause in enumerate(self.clauses)}
        inferred_order = []
        queue = deque(list(known_facts))
        
        while queue:
            p = queue.popleft()
            inferred_order.append(p)
            
            if query and p == query:
                return True, known_facts, inferred_order
                
            for i, clause in enumerate(self.clauses):
                if p in clause.premises:
                    count[i] -= 1
                    if count[i] == 0:
                        head = clause.conclusion
                        if head not in known_facts:
                            known_facts.add(head)
                            queue.append(head)
                            
        if query:
            return False, known_facts, inferred_order
        return True, known_facts, inferred_order

# Thiết lập Cơ sở tri thức chuẩn đoán lỗi mạng
fc = ForwardChainingEngine()
# Các sự kiện ban đầu ghi nhận từ người dùng
fc.add_fact("KhongVaoDuocWeb")
fc.add_fact("PingRouterThanhCong")
fc.add_fact("PingDnsServerThatBai")

# Tập các luật chuyên gia mạng
fc.add_rule(["KhongVaoDuocWeb", "PingRouterThanhCong"], "KetNoiLanTot")
fc.add_rule(["KetNoiLanTot", "PingDnsServerThatBai"], "LoiDichVuDNS")
fc.add_rule(["LoiDichVuDNS"], "CanDoiDnsSangGoogle")

query = "CanDoiDnsSangGoogle"
success, all_facts, trace = fc.infer(query)
print(f"[Suy diễn tiến] Khẳng định '{query}' có được chứng minh? -> {success}")
print(f"Toàn bộ các sự kiện được suy ra: {all_facts}")
print(f"Trình tự suy diễn: {' -> '.join(trace)}")

## 2. Động cơ Suy diễn lùi (Backward Chaining)

Suy diễn lùi bắt đầu từ mục tiêu cần chứng minh và đi ngược lại tìm kiếm các tiền đề thỏa mãn (Goal-driven).

In [ ]:
class BackwardChainingEngine:
    def __init__(self):
        self.rules = []
        self.facts = set()
        
    def add_fact(self, fact):
        self.facts.add(fact)
        
    def add_rule(self, premises, conclusion):
        self.rules.append(HornClause(premises, conclusion))
        
    def prove(self, goal, visited=None):
        if visited is None:
            visited = set()
            
        if goal in self.facts:
            return True, [f"Fact: {goal}"]
            
        if goal in visited:
            return False, [] # Tránh chu trình vô hạn
        visited.add(goal)
        
        proof_tree = []
        for rule in self.rules:
            if rule.conclusion == goal:
                # Cần chứng minh tất cả các tiền đề của luật này
                all_premises_proven = True
                sub_proofs = []
                for p in rule.premises:
                    proven, sub_p = self.prove(p, visited.copy())
                    if not proven:
                        all_premises_proven = False
                        break
                    sub_proofs.extend(sub_p)
                    
                if all_premises_proven:
                    proof_tree.extend(sub_proofs)
                    proof_tree.append(f"Rule Applied: {rule}")
                    return True, proof_tree
                    
        return False, []

bc = BackwardChainingEngine()
bc.add_fact("DenBaoRouterDo")
bc.add_fact("DayCapLanCamChac")
bc.add_rule(["DenBaoRouterDo"], "MatTinHieuQuang")
bc.add_rule(["MatTinHieuQuang", "DayCapLanCamChac"], "BaoNhaMangSuaChua")

target_goal = "BaoNhaMangSuaChua"
is_proven, proof_steps = bc.prove(target_goal)
print(f"[Suy diễn lùi] Chứng minh mục tiêu '{target_goal}': {is_proven}")
print("Cây dẫn xuất logic:")
for step in proof_steps:
    print("  *", step)

## 3. Thuật toán Hợp giải mệnh đề (Propositional Resolution Theorem Prover)

Chứng minh $KB \models \alpha$ bằng cách chỉ ra $KB \land \neg \alpha$ là mâu thuẫn (chứa mệnh đề rỗng $\square$).

In [ ]:
def resolve(ci, cj):
    """
    Hợp giải hai mệnh đề tuyển ci và cj. 
    Mỗi mệnh đề được biểu diễn dưới dạng frozenset các chuỗi literal.
    Literal phủ định có dấu tiền tố '~' (ví dụ: 'P' và '~P').
    """
    resolvents = []
    for lit in ci:
        neg_lit = lit[1:] if lit.startswith('~') else '~' + lit
        if neg_lit in cj:
            # Tạo mệnh đề mới loại bỏ lit và neg_lit
            new_clause = (ci - {lit}) | (cj - {neg_lit})
            resolvents.append(frozenset(new_clause))
    return resolvents

def resolution_solver(clauses):
    """
    Thuật toán hợp giải chứng minh phản chứng.
    """
    clauses_set = set(frozenset(c) for c in clauses)
    new_clauses = set()
    step = 0
    
    while True:
        clauses_list = list(clauses_set)
        n = len(clauses_list)
        
        for i in range(n):
            for j in range(i + 1, n):
                resolvents = resolve(clauses_list[i], clauses_list[j])
                for res in resolvents:
                    step += 1
                    if len(res) == 0:
                        # Sinh ra mệnh đề rỗng: Mâu thuẫn được tìm thấy!
                        return True, step
                    new_clauses.add(res)
                    
        if new_clauses.issubset(clauses_set):
            # Không sinh thêm được mệnh đề mới: Không thể chứng minh
            return False, step
            
        clauses_set |= new_clauses

# Ví dụ: KB = {P => Q, P}. Ta muốn chứng minh Q.
# Phủ định mục tiêu: ~Q.
# Dạng CNF:
# 1. P => Q tương đương: ~P v Q  ( {'~P', 'Q'} )
# 2. P                ( {'P'} )
# 3. ~Q               ( {'~Q'} )
clauses_example = [
    {'~P', 'Q'},
    {'P'},
    {'~Q'}
]
refuted, steps = resolution_solver(clauses_example)
print(f"[Hợp giải] Chứng minh phản chứng thành công? {refuted} (sau {steps} bước hợp giải)")

## 4. Xây dựng Đồ thị tri thức (Knowledge Graph) và Suy luận đa bước (Multi-hop Reasoning)

Biểu diễn tri thức dưới dạng các bộ ba $(h, r, t) = (\text{Head}, \text{Relation}, \text{Tail})$.

In [ ]:
class KnowledgeGraph:
    def __init__(self):
        self.triplets = []
        self.forward_index = defaultdict(lambda: defaultdict(set))
        
    def add_triplet(self, head, relation, tail):
        self.triplets.append((head, relation, tail))
        self.forward_index[head][relation].add(tail)
        
    def query(self, head, relation):
        return self.forward_index[head][relation]
        
    def multi_hop_reasoning(self, start_entity, relations_path):
        """
        Thực hiện suy luận truy vết dọc theo chuỗi quan hệ (ví dụ: ['CoCoVan', 'SinhRaTai']).
        """
        current_entities = {start_entity}
        for rel in relations_path:
            next_entities = set()
            for ent in current_entities:
                next_entities |= self.forward_index[ent][rel]
            current_entities = next_entities
            if not current_entities:
                break
        return current_entities

# Xây dựng Đồ thị tri thức về Lịch sử AI
kg = KnowledgeGraph()
kg.add_triplet("Geoffrey_Hinton", "NhanGiai", "Turing_Award")
kg.add_triplet("Yann_LeCun", "NhanGiai", "Turing_Award")
kg.add_triplet("Yoshua_Bengio", "NhanGiai", "Turing_Award")
kg.add_triplet("Geoffrey_Hinton", "HuongDan", "Ilya_Sutskever")
kg.add_triplet("Ilya_Sutskever", "DongSangLap", "OpenAI")
kg.add_triplet("OpenAI", "PhatTrien", "GPT-4")
kg.add_triplet("Yann_LeCun", "LamViecTai", "Meta_FAIR")

# Truy vấn đa bước: Người được Geoffrey_Hinton hướng dẫn đã đồng sáng lập tổ chức nào?
result_org = kg.multi_hop_reasoning("Geoffrey_Hinton", ["HuongDan", "DongSangLap"])
print(f"Suy luận 2-hop (Hinton -> HuongDan -> DongSangLap): {result_org}")

# Truy vấn 3-hop: Sản phẩm nào được phát triển bởi tổ chức mà học trò của Hinton đồng sáng lập?
result_product = kg.multi_hop_reasoning("Geoffrey_Hinton", ["HuongDan", "DongSangLap", "PhatTrien"])
print(f"Suy luận 3-hop (Hinton -> HuongDan -> DongSangLap -> PhatTrien): {result_product}")

## 5. Trực quan hóa Đồ thị Tri thức (Knowledge Graph Visualization)

In [ ]:
# Tạo bố cục vẽ đồ thị bằng matplotlib
node_positions = {
    "Geoffrey_Hinton": (1.0, 3.0),
    "Yann_LeCun": (1.0, 1.0),
    "Yoshua_Bengio": (1.0, -1.0),
    "Turing_Award": (3.0, 1.0),
    "Ilya_Sutskever": (3.0, 3.5),
    "OpenAI": (5.0, 3.5),
    "GPT-4": (7.0, 3.5),
    "Meta_FAIR": (3.0, -0.5)
}

plt.figure(figsize=(14, 7))
ax = plt.gca()

# Vẽ các cạnh quan hệ (mũi tên)
for h, r, t in kg.triplets:
    x1, y1 = node_positions[h]
    x2, y2 = node_positions[t]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color="#34495e", lw=2, shrinkA=15, shrinkB=15))
    # Ghi nhãn quan hệ ở giữa đoạn thẳng
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2 + 0.15
    ax.text(mx, my, r, fontsize=9, color="#8e44ad", ha="center", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="#d2b4de", alpha=0.9))

# Vẽ các nút thực thể
for node, (x, y) in node_positions.items():
    color = "#3498db" if "Hinton" in node or "LeCun" in node or "Bengio" in node or "Ilya" in node else "#2ecc71"
    if node in ["GPT-4", "Turing_Award"]:
        color = "#e67e22"
    ax.scatter(x, y, s=1800, c=color, edgecolors="#2c3e50", linewidth=2, zorder=4)
    ax.text(x, y, node, fontsize=9, ha="center", va="center", color="white", fontweight="bold", zorder=5)

ax.set_title("Mô hình Đồ thị Tri thức (Knowledge Graph Triplets Visualization)", fontsize=14, fontweight="bold")
ax.set_xlim(0, 8)
ax.set_ylim(-2, 4.5)
ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Tổng kết bài học thực hành

1. **Suy diễn tiến (Forward Chaining):** Xuất phát từ dữ kiện ban đầu để rút ra các hệ quả mới, tốc độ nhanh (tuyến tính) nhưng có thể sinh ra nhiều dữ kiện không liên quan đến mục tiêu.
2. **Suy diễn lùi (Backward Chaining):** Hướng mục tiêu chặt chẽ, tối ưu khi cần chứng minh hoặc gỡ lỗi một khẳng định cụ thể.
3. **Thuật toán Hợp giải (Resolution):** Cung cấp công cụ chứng minh định lý toán học hoàn chỉnh cho logic mệnh đề thông qua kỹ thuật phản chứng.
4. **Đồ thị Tri thức (Knowledge Graph):** Khắc phục độ giòn của logic mệnh đề bằng cách cấu trúc hóa quan hệ thực thể, làm nền tảng vững chắc cho các tác tử suy luận nhiều bước và các hệ thống GraphRAG tiên tiến.